# Advanced Problems with Solutions: Generator-Based Card Deck Iterables

This notebook turns a generator-based card deck into a set of advanced, runnable problems with complete solutions.

The focus is not only on generating cards, but also on the subtle distinction between **one-shot iterators** and **reusable iterables**.

## Learning goals

By the end, you should be able to:

- implement generator functions for finite collections;
- wrap generators in reusable iterable classes;
- avoid accidental generator exhaustion;
- add `__reversed__`, `__len__`, `__contains__`, and `__getitem__` correctly;
- build lazy filtering pipelines;
- deal, shuffle, slice, and query cards without unnecessary shared iterator state;
- write small assertion-based checks for iterator behavior.


## Core setup

We will use the same ordering idea throughout:

`2S, 3S, ..., AS, 2H, ..., AH, 2D, ..., AD, 2C, ..., AC`

That is: iterate suits first, then ranks within each suit.


In [1]:

from collections import namedtuple
from itertools import islice, tee
from random import Random

Card = namedtuple("Card", "rank suit")

SUITS = ("Spades", "Hearts", "Diamonds", "Clubs")
RANKS = tuple(range(2, 11)) + tuple("JQKA")

BLACK_SUITS = {"Spades", "Clubs"}
RED_SUITS = {"Hearts", "Diamonds"}
FACE_RANKS = {"J", "Q", "K", "A"}

print(Card(2, "Spades"))
print(len(SUITS), "suits")
print(len(RANKS), "ranks")


Card(rank=2, suit='Spades')
4 suits
13 ranks


---

## Problem 1 — Build the deck generator two ways

Write two generator functions that produce the exact same card order:

1. `card_gen_indexed()` should compute the suit and rank from a single numeric index.
2. `card_gen_nested()` should use nested loops.

Then prove they produce the same 52-card sequence.


In [2]:

def card_from_index(index, ranks=RANKS, suits=SUITS):
    """Return the card at a specific deck index using divmod arithmetic."""
    deck_size = len(ranks) * len(suits)

    if not 0 <= index < deck_size:
        raise IndexError(f"card index out of range: {index}")

    suit_index, rank_index = divmod(index, len(ranks))
    return Card(ranks[rank_index], suits[suit_index])


def card_gen_indexed(ranks=RANKS, suits=SUITS):
    """Generate a deck by translating each integer index into a card."""
    for index in range(len(ranks) * len(suits)):
        yield card_from_index(index, ranks, suits)


def card_gen_nested(ranks=RANKS, suits=SUITS):
    """Generate a deck using the clearest possible nested-loop version."""
    for suit in suits:
        for rank in ranks:
            yield Card(rank, suit)


indexed_cards = list(card_gen_indexed())
nested_cards = list(card_gen_nested())

assert indexed_cards == nested_cards
assert len(indexed_cards) == 52
assert indexed_cards[0] == Card(2, "Spades")
assert indexed_cards[-1] == Card("A", "Clubs")

print("First 5 cards:", indexed_cards[:5])
print("Last 5 cards: ", indexed_cards[-5:])


First 5 cards: [Card(rank=2, suit='Spades'), Card(rank=3, suit='Spades'), Card(rank=4, suit='Spades'), Card(rank=5, suit='Spades'), Card(rank=6, suit='Spades')]
Last 5 cards:  [Card(rank=10, suit='Clubs'), Card(rank='J', suit='Clubs'), Card(rank='Q', suit='Clubs'), Card(rank='K', suit='Clubs'), Card(rank='A', suit='Clubs')]


### Discussion

The indexed version is useful when you want random access by position.

The nested-loop version is usually clearer and is often the best generator implementation for simple Cartesian-product style data.


---

## Problem 2 — Demonstrate generator exhaustion

Create a generator object, consume a few items with `next()`, and then show that the same generator continues from where it left off.

Then show the correct pattern for restarting: create a new generator object.


In [3]:

one_shot_deck = card_gen_nested()

print("Manual consumption:")
print(next(one_shot_deck))
print(next(one_shot_deck))
print(next(one_shot_deck))

remaining_cards = list(one_shot_deck)
print("Remaining after consuming 3:", len(remaining_cards))
print("Trying to consume again:", list(one_shot_deck))

fresh_deck = card_gen_nested()
print("Fresh generator length:", len(list(fresh_deck)))


Manual consumption:
Card(rank=2, suit='Spades')
Card(rank=3, suit='Spades')
Card(rank=4, suit='Spades')
Remaining after consuming 3: 49
Trying to consume again: []
Fresh generator length: 52


In [4]:

def make_deck_iterator():
    """Factory function: every call returns a fresh generator object."""
    return card_gen_nested()

first_pass = list(make_deck_iterator())
second_pass = list(make_deck_iterator())

assert first_pass == second_pass
assert len(first_pass) == 52
assert len(second_pass) == 52

print("Factory pattern works: both passes are complete and equal.")


Factory pattern works: both passes are complete and equal.


### Best practice

A generator object is an **iterator**. It is one-shot.

A generator function is a **factory** for generator objects. Call the function again when you need a fresh pass.


---

## Problem 3 — Wrap the generator in a reusable iterable class

Implement a `CardDeck` class whose `__iter__` method returns a **fresh generator every time**.

Also add:

- `__len__`
- `__contains__`
- a readable `__repr__`

The object itself should be reusable.


In [5]:

class CardDeck:
    """Reusable iterable deck.

    Important design rule:
    - The deck object stores configuration only.
    - Iteration state lives inside each fresh generator returned by __iter__.
    """

    def __init__(self, ranks=RANKS, suits=SUITS):
        self.ranks = tuple(ranks)
        self.suits = tuple(suits)

    def __iter__(self):
        for suit in self.suits:
            for rank in self.ranks:
                yield Card(rank, suit)

    def __len__(self):
        return len(self.ranks) * len(self.suits)

    def __contains__(self, card):
        return isinstance(card, Card) and card.rank in self.ranks and card.suit in self.suits

    def __repr__(self):
        return f"{type(self).__name__}(ranks={self.ranks!r}, suits={self.suits!r})"


deck = CardDeck()

assert len(deck) == 52
assert Card("A", "Spades") in deck
assert Card(1, "Spades") not in deck
assert list(deck) == list(deck)  # reusable iterable

print(deck)
print("First pass first 3:", list(deck)[:3])
print("Second pass first 3:", list(deck)[:3])


CardDeck(ranks=(2, 3, 4, 5, 6, 7, 8, 9, 10, 'J', 'Q', 'K', 'A'), suits=('Spades', 'Hearts', 'Diamonds', 'Clubs'))
First pass first 3: [Card(rank=2, suit='Spades'), Card(rank=3, suit='Spades'), Card(rank=4, suit='Spades')]
Second pass first 3: [Card(rank=2, suit='Spades'), Card(rank=3, suit='Spades'), Card(rank=4, suit='Spades')]


### Why this works

Each call to `iter(deck)` calls `deck.__iter__()`, which creates a new generator frame.

So this works safely:

```python
list(deck)
list(deck)
```

because the two calls use different generator objects.


---

## Problem 4 — Prove independent iterators are truly independent

Create two iterators from the same `CardDeck` object.

Advance one iterator and prove the other iterator is not affected.


In [6]:

deck = CardDeck()

it1 = iter(deck)
it2 = iter(deck)

print("it1:", next(it1))
print("it1:", next(it1))
print("it2:", next(it2))
print("it2:", next(it2))

assert next(it1) == Card(4, "Spades")
assert next(it2) == Card(4, "Spades")

print("Both iterators advanced independently.")


it1: Card(rank=2, suit='Spades')
it1: Card(rank=3, suit='Spades')
it2: Card(rank=2, suit='Spades')
it2: Card(rank=3, suit='Spades')
Both iterators advanced independently.


In [7]:

# Nested loops over the same reusable iterable are safe.
small_deck = CardDeck(ranks=("A", "K"), suits=("Spades", "Hearts"))

pairs = []
for left in small_deck:
    for right in small_deck:
        pairs.append((left, right))

assert len(pairs) == len(small_deck) ** 2
print("Small deck size:", len(small_deck))
print("Number of ordered pairs:", len(pairs))
print("First 4 pairs:", pairs[:4])


Small deck size: 4
Number of ordered pairs: 16
First 4 pairs: [(Card(rank='A', suit='Spades'), Card(rank='A', suit='Spades')), (Card(rank='A', suit='Spades'), Card(rank='K', suit='Spades')), (Card(rank='A', suit='Spades'), Card(rank='A', suit='Hearts')), (Card(rank='A', suit='Spades'), Card(rank='K', suit='Hearts'))]


### Important warning

Nested loops are **not safe** if the object is its own iterator and stores mutable iteration state on itself.

For collection-like objects, prefer this pattern:

```python
class MyIterable:
    def __iter__(self):
        yield ...
```

instead of making the collection object also be the iterator.


---

## Problem 5 — Add reverse iteration with `__reversed__`

Implement a class that supports:

```python
reversed(deck)
```

The reverse iterator should also be one-shot, but calling `reversed(deck)` again should create a fresh reverse iterator.


In [8]:

class ReversibleCardDeck(CardDeck):
    def __reversed__(self):
        for suit in reversed(self.suits):
            for rank in reversed(self.ranks):
                yield Card(rank, suit)


deck = ReversibleCardDeck()

forward = list(deck)
backward = list(reversed(deck))

assert backward == forward[::-1]
assert backward[0] == Card("A", "Clubs")
assert backward[-1] == Card(2, "Spades")

print("Forward first 4: ", forward[:4])
print("Reverse first 4: ", backward[:4])
print("Reverse last 4:  ", backward[-4:])


Forward first 4:  [Card(rank=2, suit='Spades'), Card(rank=3, suit='Spades'), Card(rank=4, suit='Spades'), Card(rank=5, suit='Spades')]
Reverse first 4:  [Card(rank='A', suit='Clubs'), Card(rank='K', suit='Clubs'), Card(rank='Q', suit='Clubs'), Card(rank='J', suit='Clubs')]
Reverse last 4:   [Card(rank=5, suit='Spades'), Card(rank=4, suit='Spades'), Card(rank=3, suit='Spades'), Card(rank=2, suit='Spades')]


In [9]:

# reversed(deck) returns an iterator, and that iterator is one-shot.
rev_it = reversed(deck)

print(next(rev_it))
print(next(rev_it))

remaining_once = list(rev_it)
remaining_twice = list(rev_it)

print("Remaining after two next() calls:", len(remaining_once))
print("Trying same reverse iterator again:", remaining_twice)

assert len(remaining_once) == 50
assert remaining_twice == []
assert list(reversed(deck)) == list(reversed(deck))


Card(rank='A', suit='Clubs')
Card(rank='K', suit='Clubs')
Remaining after two next() calls: 50
Trying same reverse iterator again: []


### Best practice

`deck` is reusable.

`iter(deck)` and `reversed(deck)` produce iterator objects, and those iterator objects are one-shot.


---

## Problem 6 — Use `yield from` to delegate generation by suit

Refactor the card generator into smaller generators:

- `cards_of_suit(suit)`
- `deck_yield_from()`

Use `yield from` to delegate from the deck generator to each suit generator.


In [10]:

def cards_of_suit(suit, ranks=RANKS):
    """Generate all cards for exactly one suit."""
    for rank in ranks:
        yield Card(rank, suit)


def deck_yield_from(ranks=RANKS, suits=SUITS):
    """Generate a full deck by delegating to cards_of_suit."""
    for suit in suits:
        yield from cards_of_suit(suit, ranks)


assert list(deck_yield_from()) == list(CardDeck())

print("Spades packet:", list(cards_of_suit("Spades"))[:5], "...")
print("Deck first 5:", list(deck_yield_from())[:5])


Spades packet: [Card(rank=2, suit='Spades'), Card(rank=3, suit='Spades'), Card(rank=4, suit='Spades'), Card(rank=5, suit='Spades'), Card(rank=6, suit='Spades')] ...
Deck first 5: [Card(rank=2, suit='Spades'), Card(rank=3, suit='Spades'), Card(rank=4, suit='Spades'), Card(rank=5, suit='Spades'), Card(rank=6, suit='Spades')]


### When `yield from` helps

Use `yield from` when one generator naturally delegates part of its work to another generator.

It keeps the higher-level generator readable and composable.


---

## Problem 7 — Diagnose an `enumerate()` bug caused by shared iterator consumption

Create one iterator from the deck.

Wrap it with `enumerate()`.

Then consume cards directly from the original iterator before consuming from the enumerated iterator.

Explain why the index and card appear mismatched.


In [11]:

deck = CardDeck()

shared_iterator = iter(deck)
enumerated_shared_iterator = enumerate(shared_iterator)

print("Direct consumption from shared_iterator:")
print(next(shared_iterator))
print(next(shared_iterator))

print("Now consume from enumerate(shared_iterator):")
index, card = next(enumerated_shared_iterator)
print(index, card)

assert index == 0
assert card == Card(4, "Spades")


Direct consumption from shared_iterator:
Card(rank=2, suit='Spades')
Card(rank=3, suit='Spades')
Now consume from enumerate(shared_iterator):
0 Card(rank=4, suit='Spades')


### Explanation

`enumerate(shared_iterator)` does not copy the iterator.

It lazily pulls from the same iterator object.

So if the underlying iterator has already moved forward, `enumerate()` starts counting from zero while receiving whatever card is currently next.

This is why the result is:

```python
(0, Card(4, 'Spades'))
```

not:

```python
(2, Card(4, 'Spades'))
```


In [12]:

# Safer pattern: enumerate the reusable iterable directly.
for index, card in enumerate(deck):
    print(index, card)
    if index == 2:
        break

# Another safe pattern: create a fresh iterator for each independent pass.
fresh_iterator = iter(deck)
fresh_enumerated = enumerate(fresh_iterator)

assert next(fresh_enumerated) == (0, Card(2, "Spades"))
assert next(fresh_enumerated) == (1, Card(3, "Spades"))

print("Safe enumeration uses a fresh iterator.")


0 Card(rank=2, suit='Spades')
1 Card(rank=3, suit='Spades')
2 Card(rank=4, suit='Spades')
Safe enumeration uses a fresh iterator.


---

## Problem 8 — Build reusable lazy query views

Write a `CardQuery` class that wraps a reusable iterable and supports chained filters without materializing the deck.

Requirements:

- `CardQuery(deck)` should be reusable if `deck` is reusable.
- `.where(predicate)` should return a new query.
- `.take(n)` should return the first `n` matching cards.


In [13]:

class CardQuery:
    """Reusable lazy query wrapper for card iterables."""

    def __init__(self, source, predicate=None):
        self.source = source
        self.predicate = predicate or (lambda card: True)

    def __iter__(self):
        for card in self.source:
            if self.predicate(card):
                yield card

    def where(self, predicate):
        previous_predicate = self.predicate
        return CardQuery(
            self.source,
            lambda card: previous_predicate(card) and predicate(card),
        )

    def take(self, n):
        return list(islice(self, n))


def is_red(card):
    return card.suit in RED_SUITS


def is_black(card):
    return card.suit in BLACK_SUITS


def is_face(card):
    return card.rank in FACE_RANKS


query = CardQuery(CardDeck())
red_faces = query.where(is_red).where(is_face)
black_number_cards = query.where(is_black).where(lambda card: isinstance(card.rank, int))

assert len(list(red_faces)) == 8
assert len(list(red_faces)) == 8  # reusable query because source is reusable
assert red_faces.take(3) == [Card("J", "Hearts"), Card("Q", "Hearts"), Card("K", "Hearts")]
assert black_number_cards.take(4) == [Card(2, "Spades"), Card(3, "Spades"), Card(4, "Spades"), Card(5, "Spades")]

print("Red face cards:", list(red_faces))
print("First 6 black number cards:", black_number_cards.take(6))


Red face cards: [Card(rank='J', suit='Hearts'), Card(rank='Q', suit='Hearts'), Card(rank='K', suit='Hearts'), Card(rank='A', suit='Hearts'), Card(rank='J', suit='Diamonds'), Card(rank='Q', suit='Diamonds'), Card(rank='K', suit='Diamonds'), Card(rank='A', suit='Diamonds')]
First 6 black number cards: [Card(rank=2, suit='Spades'), Card(rank=3, suit='Spades'), Card(rank=4, suit='Spades'), Card(rank=5, suit='Spades'), Card(rank=6, suit='Spades'), Card(rank=7, suit='Spades')]


### Subtle limitation

`CardQuery` is reusable only if its source is reusable.

If you wrap a one-shot generator object, the query will also become effectively one-shot.


In [14]:

one_shot_query = CardQuery(card_gen_nested()).where(is_face)

first_query_pass = list(one_shot_query)
second_query_pass = list(one_shot_query)

print("First pass length:", len(first_query_pass))
print("Second pass length:", len(second_query_pass))

assert len(first_query_pass) == 16
assert second_query_pass == []


First pass length: 16
Second pass length: 0


### Safer factory-based query for one-shot sources

If your data source is a generator function, store the **factory**, not the generator object.


In [15]:

class FactoryCardQuery:
    """Reusable query that calls a source factory for every iteration."""

    def __init__(self, source_factory, predicate=None):
        self.source_factory = source_factory
        self.predicate = predicate or (lambda card: True)

    def __iter__(self):
        for card in self.source_factory():
            if self.predicate(card):
                yield card

    def where(self, predicate):
        previous_predicate = self.predicate
        return FactoryCardQuery(
            self.source_factory,
            lambda card: previous_predicate(card) and predicate(card),
        )


factory_query = FactoryCardQuery(card_gen_nested).where(is_face)

assert len(list(factory_query)) == 16
assert len(list(factory_query)) == 16

print("FactoryCardQuery is reusable even though the source is a generator function.")


FactoryCardQuery is reusable even though the source is a generator function.


---

## Problem 9 — Validate custom deck configuration

Implement a safer deck class that accepts custom ranks and suits.

Requirements:

- ranks and suits must not be empty;
- duplicates should be rejected;
- the deck should store immutable tuples internally;
- the deck should still be reusable.


In [16]:

class SafeCardDeck:
    """Reusable iterable deck with validation."""

    def __init__(self, ranks=RANKS, suits=SUITS):
        self.ranks = self._validate_unique_non_empty(ranks, "ranks")
        self.suits = self._validate_unique_non_empty(suits, "suits")

    @staticmethod
    def _validate_unique_non_empty(values, name):
        values = tuple(values)

        if not values:
            raise ValueError(f"{name} must not be empty")

        if len(set(values)) != len(values):
            raise ValueError(f"{name} must not contain duplicates: {values!r}")

        return values

    def __iter__(self):
        for suit in self.suits:
            for rank in self.ranks:
                yield Card(rank, suit)

    def __len__(self):
        return len(self.ranks) * len(self.suits)

    def __repr__(self):
        return f"{type(self).__name__}(ranks={self.ranks!r}, suits={self.suits!r})"


mini_deck = SafeCardDeck(ranks=("A", "K", "Q"), suits=("Spades", "Hearts"))

assert len(mini_deck) == 6
assert list(mini_deck) == [
    Card("A", "Spades"), Card("K", "Spades"), Card("Q", "Spades"),
    Card("A", "Hearts"), Card("K", "Hearts"), Card("Q", "Hearts"),
]

print(mini_deck)
print(list(mini_deck))


SafeCardDeck(ranks=('A', 'K', 'Q'), suits=('Spades', 'Hearts'))
[Card(rank='A', suit='Spades'), Card(rank='K', suit='Spades'), Card(rank='Q', suit='Spades'), Card(rank='A', suit='Hearts'), Card(rank='K', suit='Hearts'), Card(rank='Q', suit='Hearts')]


In [17]:

def assert_raises(expected_exception, func, *args, **kwargs):
    """Tiny helper for demonstration tests."""
    try:
        func(*args, **kwargs)
    except expected_exception as ex:
        print(f"Correctly raised {type(ex).__name__}: {ex}")
        return ex
    raise AssertionError(f"Expected {expected_exception.__name__} to be raised")


assert_raises(ValueError, SafeCardDeck, ranks=(), suits=SUITS)
assert_raises(ValueError, SafeCardDeck, ranks=("A", "A"), suits=SUITS)
assert_raises(ValueError, SafeCardDeck, ranks=RANKS, suits=("Spades", "Spades"))


Correctly raised ValueError: ranks must not be empty
Correctly raised ValueError: ranks must not contain duplicates: ('A', 'A')
Correctly raised ValueError: suits must not contain duplicates: ('Spades', 'Spades')


ValueError("suits must not contain duplicates: ('Spades', 'Spades')")

---

## Problem 10 — Add random access with `__getitem__`

Implement a sequence-like deck that supports:

```python
indexed_deck[0]
indexed_deck[-1]
indexed_deck[10:15]
```

Then use `__len__` and `__getitem__` to make Python's built-in `reversed()` fallback work even without defining `__reversed__`.


In [18]:

class IndexedCardDeck(SafeCardDeck):
    def __getitem__(self, index):
        if isinstance(index, slice):
            return [self[i] for i in range(*index.indices(len(self)))]

        if not isinstance(index, int):
            raise TypeError("deck indices must be integers or slices")

        if index < 0:
            index += len(self)

        if not 0 <= index < len(self):
            raise IndexError("deck index out of range")

        suit_index, rank_index = divmod(index, len(self.ranks))
        return Card(self.ranks[rank_index], self.suits[suit_index])

    def __iter__(self):
        for index in range(len(self)):
            yield self[index]


indexed_deck = IndexedCardDeck()

assert indexed_deck[0] == Card(2, "Spades")
assert indexed_deck[12] == Card("A", "Spades")
assert indexed_deck[13] == Card(2, "Hearts")
assert indexed_deck[-1] == Card("A", "Clubs")
assert indexed_deck[:3] == [Card(2, "Spades"), Card(3, "Spades"), Card(4, "Spades")]
assert indexed_deck[-3:] == [Card("Q", "Clubs"), Card("K", "Clubs"), Card("A", "Clubs")]
assert list(indexed_deck) == list(CardDeck())

print("Index 0:", indexed_deck[0])
print("Index 13:", indexed_deck[13])
print("Last card:", indexed_deck[-1])
print("Slice 10:15:", indexed_deck[10:15])


Index 0: Card(rank=2, suit='Spades')
Index 13: Card(rank=2, suit='Hearts')
Last card: Card(rank='A', suit='Clubs')
Slice 10:15: [Card(rank='Q', suit='Spades'), Card(rank='K', suit='Spades'), Card(rank='A', suit='Spades'), Card(rank=2, suit='Hearts'), Card(rank=3, suit='Hearts')]


In [19]:

# Because IndexedCardDeck implements __len__ and __getitem__, reversed() can use the sequence fallback.
reverse_first_five = list(islice(reversed(indexed_deck), 5))

assert reverse_first_five == [
    Card("A", "Clubs"),
    Card("K", "Clubs"),
    Card("Q", "Clubs"),
    Card("J", "Clubs"),
    Card(10, "Clubs"),
]

print(reverse_first_five)


[Card(rank='A', suit='Clubs'), Card(rank='K', suit='Clubs'), Card(rank='Q', suit='Clubs'), Card(rank='J', suit='Clubs'), Card(rank=10, suit='Clubs')]


### `__reversed__` vs sequence fallback

`reversed(obj)` works if either:

1. `obj.__reversed__()` is defined; or
2. `obj` supports the sequence protocol with `__len__` and `__getitem__`.

For a deck, direct `__reversed__` is often clearer. For indexed collections, the fallback can be convenient.


---

## Problem 11 — Write lazy `take()` and `drop()` helpers

Implement two helpers:

- `take(n, iterable)` returns a list of the first `n` items;
- `drop(n, iterable)` returns an iterator that skips the first `n` items lazily.

Use `itertools.islice`.


In [20]:

def take(n, iterable):
    """Return a list containing at most n items from iterable."""
    if n < 0:
        raise ValueError("n must be non-negative")
    return list(islice(iterable, n))


def drop(n, iterable):
    """Return an iterator that skips the first n items lazily."""
    if n < 0:
        raise ValueError("n must be non-negative")
    return islice(iterable, n, None)


assert take(4, deck) == [Card(2, "Spades"), Card(3, "Spades"), Card(4, "Spades"), Card(5, "Spades")]
assert take(3, drop(10, deck)) == [Card("Q", "Spades"), Card("K", "Spades"), Card("A", "Spades")]
assert take(100, CardDeck(ranks=("A",), suits=("Spades",))) == [Card("A", "Spades")]

print("take(5, deck):", take(5, deck))
print("take(5, drop(47, deck)):", take(5, drop(47, deck)))


take(5, deck): [Card(rank=2, suit='Spades'), Card(rank=3, suit='Spades'), Card(rank=4, suit='Spades'), Card(rank=5, suit='Spades'), Card(rank=6, suit='Spades')]
take(5, drop(47, deck)): [Card(rank=10, suit='Clubs'), Card(rank='J', suit='Clubs'), Card(rank='Q', suit='Clubs'), Card(rank='K', suit='Clubs'), Card(rank='A', suit='Clubs')]


### Iterator behavior note

`drop()` returns an iterator. If you store that iterator and consume it, it will be exhausted like any other iterator.


In [21]:

dropped = drop(50, deck)

print(list(dropped))
print(list(dropped))

assert list(drop(50, deck)) == [Card("K", "Clubs"), Card("A", "Clubs")]


[Card(rank='K', suit='Clubs'), Card(rank='A', suit='Clubs')]
[]


---

## Problem 12 — Deal cards round-robin from any iterable

Write a `deal_round_robin()` function.

Requirements:

- accepts any iterable of cards;
- returns a list of player hands;
- deals one card to each player per round;
- raises `ValueError` if there are not enough cards;
- does not require the input iterable to be a sequence.


In [22]:

def deal_round_robin(cards, players, cards_each):
    if players <= 0:
        raise ValueError("players must be positive")

    if cards_each < 0:
        raise ValueError("cards_each must be non-negative")

    iterator = iter(cards)
    hands = [[] for _ in range(players)]

    try:
        for _ in range(cards_each):
            for player_index in range(players):
                hands[player_index].append(next(iterator))
    except StopIteration as ex:
        raise ValueError("not enough cards to complete the deal") from ex

    return hands


hands = deal_round_robin(CardDeck(), players=4, cards_each=5)

assert len(hands) == 4
assert all(len(hand) == 5 for hand in hands)
assert hands[0] == [
    Card(2, "Spades"),
    Card(6, "Spades"),
    Card(10, "Spades"),
    Card("A", "Spades"),
    Card(5, "Hearts"),
]

for player_number, hand in enumerate(hands, start=1):
    print(f"Player {player_number}:", hand)


Player 1: [Card(rank=2, suit='Spades'), Card(rank=6, suit='Spades'), Card(rank=10, suit='Spades'), Card(rank='A', suit='Spades'), Card(rank=5, suit='Hearts')]
Player 2: [Card(rank=3, suit='Spades'), Card(rank=7, suit='Spades'), Card(rank='J', suit='Spades'), Card(rank=2, suit='Hearts'), Card(rank=6, suit='Hearts')]
Player 3: [Card(rank=4, suit='Spades'), Card(rank=8, suit='Spades'), Card(rank='Q', suit='Spades'), Card(rank=3, suit='Hearts'), Card(rank=7, suit='Hearts')]
Player 4: [Card(rank=5, suit='Spades'), Card(rank=9, suit='Spades'), Card(rank='K', suit='Spades'), Card(rank=4, suit='Hearts'), Card(rank=8, suit='Hearts')]


In [23]:

# The function works with one-shot generators too.
small_generator = card_gen_nested(ranks=("A", "K"), suits=("Spades", "Hearts"))
small_hands = deal_round_robin(small_generator, players=2, cards_each=2)

assert small_hands == [
    [Card("A", "Spades"), Card("A", "Hearts")],
    [Card("K", "Spades"), Card("K", "Hearts")],
]

print(small_hands)

assert_raises(ValueError, deal_round_robin, CardDeck(ranks=("A",), suits=("Spades",)), 2, 1)


[[Card(rank='A', suit='Spades'), Card(rank='A', suit='Hearts')], [Card(rank='K', suit='Spades'), Card(rank='K', suit='Hearts')]]
Correctly raised ValueError: not enough cards to complete the deal


ValueError('not enough cards to complete the deal')

---

## Problem 13 — Return hands and the undealt remainder

Modify the dealing function so it returns both:

1. the hands;
2. an iterator for the remaining undealt cards.

This is useful when you want to continue consuming the same deck stream after dealing.


In [24]:

def deal_with_remainder(cards, players, cards_each):
    if players <= 0:
        raise ValueError("players must be positive")

    if cards_each < 0:
        raise ValueError("cards_each must be non-negative")

    iterator = iter(cards)
    hands = [[] for _ in range(players)]

    try:
        for _ in range(cards_each):
            for player_index in range(players):
                hands[player_index].append(next(iterator))
    except StopIteration as ex:
        raise ValueError("not enough cards to complete the deal") from ex

    return hands, iterator


hands, remainder = deal_with_remainder(CardDeck(), players=4, cards_each=5)
remainder_preview = take(5, remainder)

assert len(hands) == 4
assert all(len(hand) == 5 for hand in hands)
assert remainder_preview == [Card(9, "Hearts"), Card(10, "Hearts"), Card("J", "Hearts"), Card("Q", "Hearts"), Card("K", "Hearts")]

print("First hand:", hands[0])
print("Next 5 undealt cards:", remainder_preview)
print("Cards still left after preview:", len(list(remainder)))


First hand: [Card(rank=2, suit='Spades'), Card(rank=6, suit='Spades'), Card(rank=10, suit='Spades'), Card(rank='A', suit='Spades'), Card(rank=5, suit='Hearts')]
Next 5 undealt cards: [Card(rank=9, suit='Hearts'), Card(rank=10, suit='Hearts'), Card(rank='J', suit='Hearts'), Card(rank='Q', suit='Hearts'), Card(rank='K', suit='Hearts')]
Cards still left after preview: 27


### Design note

Returning the remainder as an iterator is memory-efficient.

But remember: it is the same continuing stream. Once consumed, it cannot be restarted.


---

## Problem 14 — Build a shuffled iterable deck

Implement a `ShuffledDeck` that shuffles a fresh list of cards inside `__iter__`.

Requirements:

- it should be reusable;
- with a fixed seed, every pass should produce the same shuffled order;
- without a fixed seed, every pass may produce a different order;
- do not mutate the original ordered deck configuration.


In [25]:

class ShuffledDeck(CardDeck):
    def __init__(self, ranks=RANKS, suits=SUITS, seed=None):
        super().__init__(ranks=ranks, suits=suits)
        self.seed = seed

    def __iter__(self):
        cards = list(super().__iter__())
        rng = Random(self.seed)
        rng.shuffle(cards)
        yield from cards

    def __repr__(self):
        return f"{type(self).__name__}(ranks={self.ranks!r}, suits={self.suits!r}, seed={self.seed!r})"


shuffled = ShuffledDeck(seed=42)
first_shuffle = list(shuffled)
second_shuffle = list(shuffled)
ordered = list(CardDeck())

assert first_shuffle == second_shuffle
assert set(first_shuffle) == set(ordered) and len(first_shuffle) == len(ordered)
assert first_shuffle != ordered

print(shuffled)
print("First 10 shuffled cards:", first_shuffle[:10])


ShuffledDeck(ranks=(2, 3, 4, 5, 6, 7, 8, 9, 10, 'J', 'Q', 'K', 'A'), suits=('Spades', 'Hearts', 'Diamonds', 'Clubs'), seed=42)
First 10 shuffled cards: [Card(rank='J', suit='Spades'), Card(rank='Q', suit='Hearts'), Card(rank='A', suit='Hearts'), Card(rank=5, suit='Spades'), Card(rank=10, suit='Hearts'), Card(rank='A', suit='Diamonds'), Card(rank=5, suit='Hearts'), Card(rank=2, suit='Clubs'), Card(rank=8, suit='Hearts'), Card(rank='K', suit='Spades')]


### Variant: changing shuffle on each pass

The previous implementation is deterministic when a seed is supplied.

Sometimes you want a reusable object where each pass produces a different shuffle. In that case, store a `Random` instance on the object and advance its state each time `__iter__` is called.


In [26]:

class AdvancingShuffledDeck(CardDeck):
    def __init__(self, ranks=RANKS, suits=SUITS, seed=None):
        super().__init__(ranks=ranks, suits=suits)
        self._rng = Random(seed)

    def __iter__(self):
        cards = list(super().__iter__())
        self._rng.shuffle(cards)
        yield from cards


advancing = AdvancingShuffledDeck(seed=42)
shuffle_a = list(advancing)
shuffle_b = list(advancing)

assert shuffle_a != shuffle_b
assert set(shuffle_a) == set(shuffle_b) == set(CardDeck()) and len(shuffle_a) == len(shuffle_b) == len(CardDeck())

print("First pass first 5: ", shuffle_a[:5])
print("Second pass first 5:", shuffle_b[:5])


First pass first 5:  [Card(rank='J', suit='Spades'), Card(rank='Q', suit='Hearts'), Card(rank='A', suit='Hearts'), Card(rank=5, suit='Spades'), Card(rank=10, suit='Hearts')]
Second pass first 5: [Card(rank=2, suit='Hearts'), Card(rank=6, suit='Clubs'), Card(rank=4, suit='Hearts'), Card(rank='J', suit='Spades'), Card(rank='A', suit='Hearts')]


---

## Problem 15 — Avoid accidental materialization in membership checks

For a normal iterable-only deck, `card in deck` may iterate until it finds the card.

Implement a direct `__contains__` method for a validated deck so membership does not have to scan all generated cards.


In [27]:

class FastContainsDeck(SafeCardDeck):
    def __init__(self, ranks=RANKS, suits=SUITS):
        super().__init__(ranks=ranks, suits=suits)
        self._rank_set = set(self.ranks)
        self._suit_set = set(self.suits)

    def __contains__(self, card):
        return isinstance(card, Card) and card.rank in self._rank_set and card.suit in self._suit_set


fast_deck = FastContainsDeck()

assert Card("A", "Clubs") in fast_deck
assert Card("Joker", "Black") not in fast_deck
assert "not a card" not in fast_deck

print(Card("A", "Clubs") in fast_deck)
print(Card("Joker", "Black") in fast_deck)


True
False


### Best practice

When membership can be answered from stored configuration, implement `__contains__` directly.

This avoids consuming an iterator and avoids scanning generated values.


---

## Problem 16 — Use `itertools.tee()` carefully

`tee()` can split one iterator into multiple iterators.

Use it to create two streams from a generator, then explain the tradeoff.


In [28]:

source = card_gen_nested()
stream_a, stream_b = tee(source, 2)

print("A gets:", next(stream_a))
print("A gets:", next(stream_a))
print("B gets:", next(stream_b))
print("B gets:", next(stream_b))

assert next(stream_a) == Card(4, "Spades")
assert next(stream_b) == Card(4, "Spades")

print("tee() created two logical streams from one original generator.")


A gets: Card(rank=2, suit='Spades')
A gets: Card(rank=3, suit='Spades')
B gets: Card(rank=2, suit='Spades')
B gets: Card(rank=3, suit='Spades')
tee() created two logical streams from one original generator.


### `tee()` tradeoff

`tee()` is useful, but it may need to cache items internally if one derived iterator gets far ahead of another.

For finite small decks this is fine.

For huge or infinite streams, prefer designing the source as a reusable iterable or factory when possible.


---

## Problem 17 — Build a rank-value scoring pipeline

Create helpers that assign numeric values to cards and lazily filter high-value cards.

Requirements:

- number cards keep their numeric value;
- Jack = 11, Queen = 12, King = 13, Ace = 14;
- `high_cards(cards, minimum)` should be lazy.


In [29]:

RANK_VALUES = {rank: rank for rank in range(2, 11)} | {"J": 11, "Q": 12, "K": 13, "A": 14}


def card_value(card):
    return RANK_VALUES[card.rank]


def high_cards(cards, minimum=11):
    for card in cards:
        if card_value(card) >= minimum:
            yield card


aces = list(high_cards(CardDeck(), minimum=14))
face_or_ace_first_six = take(6, high_cards(CardDeck(), minimum=11))

assert aces == [Card("A", "Spades"), Card("A", "Hearts"), Card("A", "Diamonds"), Card("A", "Clubs")]
assert face_or_ace_first_six == [Card("J", "Spades"), Card("Q", "Spades"), Card("K", "Spades"), Card("A", "Spades"), Card("J", "Hearts"), Card("Q", "Hearts")]

print("Aces:", aces)
print("First 6 face-or-ace cards:", face_or_ace_first_six)


Aces: [Card(rank='A', suit='Spades'), Card(rank='A', suit='Hearts'), Card(rank='A', suit='Diamonds'), Card(rank='A', suit='Clubs')]
First 6 face-or-ace cards: [Card(rank='J', suit='Spades'), Card(rank='Q', suit='Spades'), Card(rank='K', suit='Spades'), Card(rank='A', suit='Spades'), Card(rank='J', suit='Hearts'), Card(rank='Q', suit='Hearts')]


### Lazy pipeline example

This expression does not build the full list of high cards before slicing:

```python
take(6, high_cards(CardDeck(), minimum=11))
```

`take()` pulls only as many matching values as needed.


---

## Problem 18 — Design a final production-style deck class

Combine the best ideas into one class.

Requirements:

- validated ranks and suits;
- reusable forward iteration;
- reusable reverse iteration;
- length;
- fast membership;
- integer indexing;
- slicing;
- `.shuffled(seed=...)` method returning a shuffled iterable view.


In [30]:

class ProductionCardDeck:
    def __init__(self, ranks=RANKS, suits=SUITS):
        self.ranks = SafeCardDeck._validate_unique_non_empty(ranks, "ranks")
        self.suits = SafeCardDeck._validate_unique_non_empty(suits, "suits")
        self._rank_set = set(self.ranks)
        self._suit_set = set(self.suits)

    def __len__(self):
        return len(self.ranks) * len(self.suits)

    def __iter__(self):
        for suit in self.suits:
            for rank in self.ranks:
                yield Card(rank, suit)

    def __reversed__(self):
        for suit in reversed(self.suits):
            for rank in reversed(self.ranks):
                yield Card(rank, suit)

    def __contains__(self, card):
        return isinstance(card, Card) and card.rank in self._rank_set and card.suit in self._suit_set

    def __getitem__(self, index):
        if isinstance(index, slice):
            return [self[i] for i in range(*index.indices(len(self)))]

        if not isinstance(index, int):
            raise TypeError("deck indices must be integers or slices")

        if index < 0:
            index += len(self)

        if not 0 <= index < len(self):
            raise IndexError("deck index out of range")

        suit_index, rank_index = divmod(index, len(self.ranks))
        return Card(self.ranks[rank_index], self.suits[suit_index])

    def shuffled(self, seed=None):
        return ShuffledDeck(ranks=self.ranks, suits=self.suits, seed=seed)

    def __repr__(self):
        return f"{type(self).__name__}(ranks={self.ranks!r}, suits={self.suits!r})"


prod = ProductionCardDeck()

assert len(prod) == 52
assert list(prod)[:3] == [Card(2, "Spades"), Card(3, "Spades"), Card(4, "Spades")]
assert list(reversed(prod))[:3] == [Card("A", "Clubs"), Card("K", "Clubs"), Card("Q", "Clubs")]
assert prod[0] == Card(2, "Spades")
assert prod[-1] == Card("A", "Clubs")
assert prod[10:13] == [Card("Q", "Spades"), Card("K", "Spades"), Card("A", "Spades")]
assert Card("A", "Hearts") in prod
assert Card("Joker", "Black") not in prod
assert set(prod.shuffled(seed=123)) == set(prod) and len(list(prod.shuffled(seed=123))) == len(prod)

print(prod)
print("First 5:", prod[:5])
print("Last 5 reversed:", list(reversed(prod))[:5])
print("Shuffled first 5:", list(prod.shuffled(seed=123))[:5])


ProductionCardDeck(ranks=(2, 3, 4, 5, 6, 7, 8, 9, 10, 'J', 'Q', 'K', 'A'), suits=('Spades', 'Hearts', 'Diamonds', 'Clubs'))
First 5: [Card(rank=2, suit='Spades'), Card(rank=3, suit='Spades'), Card(rank=4, suit='Spades'), Card(rank=5, suit='Spades'), Card(rank=6, suit='Spades')]
Last 5 reversed: [Card(rank='A', suit='Clubs'), Card(rank='K', suit='Clubs'), Card(rank='Q', suit='Clubs'), Card(rank='J', suit='Clubs'), Card(rank=10, suit='Clubs')]
Shuffled first 5: [Card(rank='K', suit='Diamonds'), Card(rank='Q', suit='Clubs'), Card(rank=5, suit='Diamonds'), Card(rank='J', suit='Clubs'), Card(rank=9, suit='Hearts')]


---

## Final checklist: generator-to-iterable best practices

Use this checklist when designing iterable objects from generators:

1. **Generator object = one-shot iterator.** Do not expect it to restart.
2. **Generator function = iterator factory.** Call it again for a fresh pass.
3. **Reusable iterable class = stores data/configuration, not iteration progress.**
4. **`__iter__` should return a fresh iterator each time.** A generator function body inside `__iter__` is a clean way to do that.
5. **Do not share one iterator between unrelated consumers** unless you intend them to share progress.
6. **`enumerate()`, `map()`, `filter()`, `zip()`, and many itertools tools are lazy.** They do not copy their inputs.
7. **Implement `__reversed__`** when reverse order is natural and efficient.
8. **Implement `__len__`, `__contains__`, and `__getitem__`** when they are meaningful and can be done efficiently.
9. **Use `yield from`** to delegate to subgenerators cleanly.
10. **Write small assertions** to prove reusability, order, exhaustion behavior, and edge cases.
